In [ ]:
# Install the required Python packages:
# - langchain: the core LangChain framework for building AI-powered applications
# - langchain-openai: LangChain's integration with OpenAI models (e.g., GPT)
!pip install -q langchain langchain-openai


In [ ]:
# Import Google Colab utilities to access secrets stored in the Colab environment
from google.colab import userdata

# Import the agent factory function from LangChain
# create_agent builds a ready-to-use AI agent that can reason and call tools automatically
from langchain.agents import create_agent

# HumanMessage represents a message sent by the user in the conversation
from langchain.messages import HumanMessage

# The @tool decorator used to register Python functions as tools an agent can call
from langchain.tools import tool

# BaseMessage is the common base type for all message types (human, AI, system, tool)
from langchain_core.messages import BaseMessage

# ChatOpenAI is LangChain's wrapper around OpenAI's chat models (e.g., GPT-4)
from langchain_openai import ChatOpenAI

# SecretStr wraps sensitive strings so they are not accidentally printed in plain text
from pydantic import SecretStr
from typing import List

# Retrieve the OpenAI API key from Google Colab's secret storage and wrap it securely
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper function to print every message in a conversation in a readable format
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        # pretty_print() formats each message showing its role (Human/AI/Tool) and content
        message.pretty_print()


In [ ]:
# The @tool decorator turns a regular Python function into a LangChain "tool".
# A tool is a capability that the AI agent can decide to call during a conversation
# when it needs specific information it cannot reason about on its own.
@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    # Hard-coded lookup table mapping artist names (lowercase) to their upcoming venue
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    # Normalize the input to lowercase and strip whitespace before looking it up
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    # Hard-coded route table mapping (origin, destination) tuples to travel time strings
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    # Build the lookup key by normalizing both city names
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")


In [ ]:
# Create an AI agent by combining:
# - model: the LLM (Large Language Model) that powers the agent's reasoning
# - tools: the Python functions the agent is allowed to call
# - system_prompt: a standing instruction that shapes the agent's personality and behavior
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[lookup_tour_stop, estimate_drive_time],
    # The system prompt tells the agent its role. It will follow this throughout the conversation.
    system_prompt="You are a practical live-music concierge. Use tools when they help you give a precise answer."
)


In [ ]:
# Invoke the agent with a user question.
# The agent will internally:
#   1. Read the human message
#   2. Decide which tools to call (lookup_tour_stop, estimate_drive_time)
#   3. Call those tools and collect results
#   4. Compose a final answer for the user
# The result dictionary contains "messages" — the full conversation history including tool calls.
plan_concert_trip = agent.invoke(
    input={
      "messages": [HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time.")]
    }
)


In [ ]:
# Print all messages from the concert trip interaction.
# This shows the full flow: user question → tool calls → tool results → AI final answer.
print_conversation(plan_concert_trip["messages"])


In [ ]:
# Test whether the agent has memory of the previous conversation.
# By default, create_agent does NOT persist memory between separate .invoke() calls —
# each invocation starts with a fresh, empty message history.
# This call demonstrates that the agent has no recollection of the earlier concert trip query.
test_agent_memory = agent.invoke(
    input={
        "messages": [HumanMessage("What do you remember about me?")]
    }
)


In [ ]:
# Print the memory-test conversation to see that the agent responds without any prior context.
print_conversation(test_agent_memory["messages"])
